In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root:", project_root)

Project root: c:\Users\ASUS\Desktop\NLP-News-Intelligence


In [3]:
from src.pytorch.model import NewsClassifier

print("NewsClassifier imported successfully!")

NewsClassifier imported successfully!


In [4]:
from src.pytorch.model import NewsClassifier

print("NewsClassifier imported successfully!")

NewsClassifier imported successfully!


In [5]:
import json
import torch

from src.pytorch.model import NewsClassifier

with open(
    "../data/processed/vocab.json",
    "r",
    encoding="utf-8"
) as f:
    vocab = json.load(f)

model = NewsClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=4,
    dropout=0.3
)

model.load_state_dict(
    torch.load(
        "../models/pytorch/best_news_classifier.pt",
        map_location="cpu"
    )
)

model.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [7]:
from pathlib import Path

project_root = Path.cwd().parent

print("Project root:")
print(project_root)

print("\nFiles inside src/preprocessing:")
for file in (project_root / "src" / "preprocessing").iterdir():
    print(file.name)

Project root:
c:\Users\ASUS\Desktop\NLP-News-Intelligence

Files inside src/preprocessing:
cleaning.py
__pycache__


In [10]:
from src.preprocessing.tokenization import (
    text_to_sequence,
    pad_sequence
)

print("Tokenization functions imported successfully!")

Tokenization functions imported successfully!


In [12]:
from datasets import load_dataset

dataset = load_dataset("SetFit/ag_news")

print(dataset)

c:\Users\ASUS\Desktop\NLP-News-Intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 7600
    })
})


In [13]:
test_data = dataset["test"]

print("Test samples:", len(test_data))

Test samples: 7600


In [14]:
import torch

predictions = []
actual_labels = []

model.eval()

for item in test_data:
    sequence = text_to_sequence(item["text"], vocab)
    sequence = pad_sequence(sequence, 100)

    input_tensor = torch.tensor(
        [sequence],
        dtype=torch.long
    )

    with torch.no_grad():
        output = model(input_tensor)
        predicted_class = torch.argmax(output, dim=1).item()

    predictions.append(predicted_class)
    actual_labels.append(item["label"])

print("Predictions:", len(predictions))
print("Actual labels:", len(actual_labels))

Predictions: 7600
Actual labels: 7600


In [15]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(actual_labels, predictions)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 0.6729
Test Accuracy: 67.29%


In [16]:
from src.preprocessing.cleaning import clean_text

print("Original:")
print(test_data[0]["text"])

print("\nCleaned:")
print(clean_text(test_data[0]["text"]))

Original:
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.

Cleaned:
fears for t n pension after talks unions representing workers at turner newall say they are disappointed after talks with stricken parent firm federal mogul


In [17]:
import torch

predictions = []
actual_labels = []

model.eval()

for item in test_data:
    cleaned_text = clean_text(item["text"])

    sequence = text_to_sequence(cleaned_text, vocab)
    sequence = pad_sequence(sequence, 100)

    input_tensor = torch.tensor(
        [sequence],
        dtype=torch.long
    )

    with torch.no_grad():
        output = model(input_tensor)
        predicted_class = torch.argmax(output, dim=1).item()

    predictions.append(predicted_class)
    actual_labels.append(item["label"])

print("Predictions:", len(predictions))
print("Actual labels:", len(actual_labels))

Predictions: 7600
Actual labels: 7600


In [18]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(actual_labels, predictions)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 0.9118
Test Accuracy: 91.18%


In [19]:
from sklearn.metrics import classification_report

label_names = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

report = classification_report(
    actual_labels,
    predictions,
    target_names=label_names,
    digits=2
)

print(report)

              precision    recall  f1-score   support

       World       0.93      0.90      0.92      1900
      Sports       0.95      0.98      0.96      1900
    Business       0.89      0.86      0.88      1900
    Sci/Tech       0.88      0.91      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



In [20]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(actual_labels, predictions)

cm_df = pd.DataFrame(
    cm,
    index=label_names,
    columns=label_names
)

print(cm_df)

          World  Sports  Business  Sci/Tech
World      1707      60        67        66
Sports       21    1853        15        11
Business     62      29      1642       167
Sci/Tech     39      15       118      1728


In [21]:
from pathlib import Path
import pandas as pd

results_dir = Path("../results/metrics")
results_dir.mkdir(parents=True, exist_ok=True)

verification = pd.DataFrame({
    "Metric": [
        "Test Samples",
        "Test Accuracy"
    ],
    "Value": [
        len(actual_labels),
        accuracy
    ]
})

verification.to_csv(
    results_dir / "pytorch_saved_model_verification.csv",
    index=False
)

print("Saved:")
print(results_dir / "pytorch_saved_model_verification.csv")

Saved:
..\results\metrics\pytorch_saved_model_verification.csv


In [22]:
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

keras_model_path = Path("../models/keras/best_news_classifier.keras")

keras_model = keras.models.load_model(keras_model_path)

print("Keras model loaded successfully!")
print("TensorFlow version:", tf.__version__)

Keras model loaded successfully!
TensorFlow version: 2.21.0


In [23]:
import numpy as np

keras_test_sequences = []

for item in test_data:
    cleaned_text = clean_text(item["text"])

    sequence = text_to_sequence(cleaned_text, vocab)
    sequence = pad_sequence(sequence, 100)

    keras_test_sequences.append(sequence)

X_test_keras = np.array(keras_test_sequences)

y_test_keras = np.array(actual_labels)

print("X test shape:", X_test_keras.shape)
print("Y test shape:", y_test_keras.shape)

X test shape: (7600, 100)
Y test shape: (7600,)


In [24]:
keras_probabilities = keras_model.predict(
    X_test_keras,
    batch_size=64,
    verbose=1
)

keras_predictions = np.argmax(
    keras_probabilities,
    axis=1
)

print("Predictions:", len(keras_predictions))
print("Actual labels:", len(y_test_keras))

119/119 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step
Predictions: 7600
Actual labels: 7600


In [25]:
from sklearn.metrics import accuracy_score

keras_accuracy = accuracy_score(
    y_test_keras,
    keras_predictions
)

print(f"Keras Test Accuracy: {keras_accuracy:.4f}")
print(f"Keras Test Accuracy: {keras_accuracy * 100:.2f}%")

Keras Test Accuracy: 0.9138
Keras Test Accuracy: 91.38%


In [26]:
from sklearn.metrics import classification_report

keras_report = classification_report(
    y_test_keras,
    keras_predictions,
    target_names=label_names,
    digits=2
)

print(keras_report)

              precision    recall  f1-score   support

       World       0.92      0.91      0.92      1900
      Sports       0.96      0.98      0.97      1900
    Business       0.88      0.88      0.88      1900
    Sci/Tech       0.89      0.89      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



In [27]:
from sklearn.metrics import confusion_matrix
import pandas as pd

keras_cm = confusion_matrix(
    y_test_keras,
    keras_predictions
)

keras_cm_df = pd.DataFrame(
    keras_cm,
    index=label_names,
    columns=label_names
)

print(keras_cm_df)

          World  Sports  Business  Sci/Tech
World      1731      45        72        52
Sports       23    1853        16         8
Business     68      16      1675       141
Sci/Tech     60      19       135      1686


In [28]:
from pathlib import Path
import pandas as pd

results_dir = Path("../results/metrics")
results_dir.mkdir(parents=True, exist_ok=True)

keras_verification = pd.DataFrame({
    "Metric": [
        "Test Samples",
        "Test Accuracy"
    ],
    "Value": [
        len(y_test_keras),
        keras_accuracy
    ]
})

keras_verification.to_csv(
    results_dir / "keras_saved_model_verification.csv",
    index=False
)

print("Saved:")
print(results_dir / "keras_saved_model_verification.csv")

Saved:
..\results\metrics\keras_saved_model_verification.csv
